# 🗄️ Partie 5 : Création du Data Warehouse (AWS RDS)

## 🎯 Objectif
Structurer les données brutes pour les rendre exploitables. Nous passons de fichiers plats (CSV) à une base de données relationnelle SQL hébergée dans le Cloud.

## 🛠 Choix Techniques
* **AWS RDS (PostgreSQL)** : Base de données managée robuste et performante pour les requêtes analytiques.
* **SQLAlchemy** : Utilisation d'un ORM (Object Relational Mapper) pour insérer les DataFrames Pandas directement en base.
    * *Avantage* : Gestion automatique des types de données (Schema) et sécurité accrue par rapport aux requêtes SQL brutes.

## 📐 Processus ETL
* **Extract** : Lecture des CSV **depuis AWS S3** (et non en local) — conformément au schéma Data Lake → ETL → Data Warehouse.
* **Transform** : Nettoyage des types, conversion des scores, gestion des valeurs manquantes.
* **Load** : Insertion dans PostgreSQL via `to_sql()`.

## 🔑 Note sur le `city_id`
Le champ `city_id` (identifiant unique par ville, demandé par l'énoncé) est généré dans le notebook 01 et propagé dans tous les CSV en amont. Il arrive automatiquement dans les tables SQL via `to_sql()`, ce qui permet des jointures fiables entre les tables `weather` et `hotels` sans ambiguïté sur les noms de villes.

In [1]:
import pandas as pd
import boto3
import os
import io
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# 1. Chargement de la configuration
load_dotenv()

# AWS S3 (source : Data Lake)
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
BUCKET_NAME = os.getenv("AWS_BUCKET_NAME")

# AWS RDS (destination : Data Warehouse)
DB_ENDPOINT = os.getenv("DB_HOST")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_USER = os.getenv("DB_USER")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# --- EXTRACT : Lecture depuis S3 (Data Lake) ---
print("📥 EXTRACT : Lecture des CSV depuis S3...")

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name="eu-west-3"
)
s3 = session.client("s3")

def read_csv_from_s3(bucket, key):
    """Télécharge un CSV depuis S3 et le charge dans un DataFrame."""
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

df_weather = read_csv_from_s3(BUCKET_NAME, 'cities_weather_data_7days.csv')
print(f"   ✅ weather : {len(df_weather)} lignes lues depuis s3://{BUCKET_NAME}/cities_weather_data_7days.csv")

df_hotels = read_csv_from_s3(BUCKET_NAME, 'hotels_data.csv')
print(f"   ✅ hotels  : {len(df_hotels)} lignes lues depuis s3://{BUCKET_NAME}/hotels_data.csv")

# --- TRANSFORM : Nettoyage ---
print("\n🔧 TRANSFORM : Nettoyage des données...")

# Conversion du score en numérique (certains sont des strings "8.28")
df_hotels['score'] = pd.to_numeric(df_hotels['score'], errors='coerce')
print(f"   ✅ Scores convertis en float ({df_hotels['score'].isna().sum()} valeurs manquantes)")

# --- LOAD : Insertion dans RDS (Data Warehouse) ---
conn_str = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_ENDPOINT}:{DB_PORT}/{DB_NAME}"

try:
    print(f"\n📤 LOAD : Connexion à RDS ({DB_ENDPOINT})...")
    engine = create_engine(conn_str)
    
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
        print("   ✅ Connexion réussie !")

    # Insertion des tables
    df_weather.to_sql('weather', engine, if_exists='replace', index=False)
    print(f"   ✅ Table 'weather' : {len(df_weather)} lignes insérées.")
    
    df_hotels.to_sql('hotels', engine, if_exists='replace', index=False)
    print(f"   ✅ Table 'hotels'  : {len(df_hotels)} lignes insérées.")

    print("\n🎉 ETL terminé : S3 → Transform → RDS PostgreSQL")

except Exception as e:
    print(f"\n❌ Erreur critique : {e}")

📥 EXTRACT : Lecture des CSV depuis S3...
   ✅ weather : 35 lignes lues depuis s3://kayak-project-aws/cities_weather_data_7days.csv
   ✅ hotels  : 700 lignes lues depuis s3://kayak-project-aws/hotels_data.csv

🔧 TRANSFORM : Nettoyage des données...
   ✅ Scores convertis en float (8 valeurs manquantes)

📤 LOAD : Connexion à RDS (kayak-db.cnssuskuy2g7.eu-north-1.rds.amazonaws.com)...
   ✅ Connexion réussie !
   ✅ Table 'weather' : 35 lignes insérées.
   ✅ Table 'hotels'  : 700 lignes insérées.

🎉 ETL terminé : S3 → Transform → RDS PostgreSQL
